# Глубинное обучение, магистратура НИУ ВШЭ

## Домашнее задание 3. Трансформеры

### Общая информация

Мягкий дедлайн: 14.12.2026 00:00

Жесткий дедлайн: 21.12.2026 00:00

### О задании

В этом задании вам предстоит добавить к BERT декодер-трансформер и решить задачу суммаризации: писать короткое tl;dr для новостей на русском языке.

Задание состоит из нескольких частей:
1. Подготовка данных.
2. Реализация декодера и генерации.
3. Обучение модели и метрики качества.
4. Менее жадные стратегии выбора следующего токена: top-k, top-p и beam search.

Для обучения понадобится GPU, подойдут Google Colab или Kaggle.

### Оценивание и штрафы

Каждая из задач имеет определенную «стоимость» (указана в скобках около задачи). Максимально допустимая оценка за работу без бонуса 10 баллов, за бонус можно получить еще 1 балл.

Сдавать задание после жесткого дедлайна нельзя. Сдача после мягкого дедлайна штрафуется ступенчато: -1 балл за каждые сутки опоздания. Дважды за курс можно сдать работу в жесткий дедлайн без штрафа, заранее предупредив ассистента.

Задание выполняется самостоятельно. «Похожие» решения считаются плагиатом, и все задействованные студенты (в том числе те, у кого списали) не могут получить за него больше 0 баллов. Если вы нашли решение какого-то из заданий (или его часть) в открытом источнике, укажите ссылку на этот источник в отдельном блоке в конце работы (скорее всего, вы будете не единственным, кто это нашел, и ссылка снимет подозрение в плагиате).

Неэффективная реализация кода может негативно отразиться на оценке. Оценка также может быть снижена за плохо читаемый код и плохо оформленные графики. Все ответы должны сопровождаться кодом или комментариями о том, как они были получены.

Использование генеративных моделей допустимо на следующих условиях:
- код, написанный генеративными моделями, составляет не более 30% решения;
- указаны использованная модель и промпты;
- в конце работы описан ваш опыт: как часто приходилось исправлять код руками или просить модель что-то исправить, было ли это быстрее, чем написать код самим.

Если эти условия не выполнены, работа оценивается в 0 баллов.

### Формат сдачи

Сдайте выполненный ноутбук с сохраненными выводами ячеек. Назовите файл `HW3-Username.ipynb`, где Username ваши фамилия и имя на латинице именно в таком порядке (например, `HW3-IvanovIvan.ipynb`).

In [ ]:
!pip install transformers datasets evaluate rouge_score bert_score

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from datasets import load_dataset
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, BertModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0xDEC0DE)

## Часть 1. Подготовка данных (0.5 балла)

Мы воспользуемся датасетом [gazeta](https://huggingface.co/datasets/IlyaGusev/gazeta) Ильи Гусева. Он состоит из пар «полный текст новости, ее краткое содержание».

Можно взять больше или меньше данных, но что-то адекватное обычно получается начиная с 10% обучающей выборки.

In [ ]:
train_data = load_dataset("IlyaGusev/gazeta", split="train[:10%]")
val_data = load_dataset("IlyaGusev/gazeta", split="validation[:10%]")
train_data, val_data

**Задание 1 (0.5 балла).** Тексты перед подачей в модель нужно **токенизировать**. Напишите функцию `preprocess`, которая возвращает `input_ids` и `attention_mask` для текста новости и `labels` для краткого содержания.

- Тексты дополняйте паддингом до `max_length=512`, краткие содержания до `max_length=128`.
- Обрезайте тексты, длина которых в токенах превышает `max_length`.
- Токенизатор BERT добавляет в начало `[CLS]`, а в конец `[SEP]`. Будем использовать их как токены начала и конца последовательности в декодере.

Соберите `DataLoader` для обучения и валидации.

In [ ]:
model_name = "deepvk/bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

BOS_ID = tokenizer.cls_token_id
EOS_ID = tokenizer.sep_token_id
PAD_ID = tokenizer.pad_token_id


def preprocess(examples):
    # YOUR CODE HERE (－.－)...zzzZZZzzzZZZ
    model_inputs = ...
    return model_inputs

In [ ]:
columns = ["input_ids", "attention_mask", "labels"]
train_tokenized = train_data.map(preprocess, batched=True)
val_tokenized = val_data.map(preprocess, batched=True)
train_tokenized.set_format("torch", columns=columns)
val_tokenized.set_format("torch", columns=columns)

In [ ]:
train_dataloader = ...  # YOUR CODE HERE
eval_dataloader = ...  # YOUR CODE HERE

## Часть 2. Декодер (3 балла)

**Задание 2 (3 балла).** Реализуйте собственный декодер для генерации текста. Энкодером будет предобученный BERT, его выходы `last_hidden_state` декодер получает как `memory`.

Что нужно сделать:
- эмбеддинги токенов и позиционные эмбеддинги для входа декодера (без информации о позиции декодер почти не отличает порядок токенов);
- стек слоев `nn.TransformerDecoderLayer` (или свои слои, если хочется), используйте `batch_first=True`;
- линейный слой из `hidden_size` в размер словаря;
- маску `generate_square_subsequent_mask`, которая запрещает токену смотреть на следующие за ним;
- маски паддинга: для `memory` по `attention_mask` энкодера и для входа декодера по `PAD_ID`;
- жадную генерацию в `generate`: начинаем с `BOS_ID`, на каждом шаге добавляем самый вероятный токен и останавливаемся на `EOS_ID` или по `max_len`.

`forward` должен возвращать **логиты** размера `(B, T, vocab_size)`, без softmax: `nn.CrossEntropyLoss` применяет его сам.

Можете вдохновляться кодом с семинара. Подумайте об инициализации весов. Энкодер можно заморозить (`freeze_encoder=True`), чтобы обучение шло быстрее; если размораживаете, возьмите для него меньший learning rate.

In [ ]:
class BertSummarizer(nn.Module):
    def __init__(
        self,
        bert_model_name=model_name,
        num_decoder_layers=3,
        num_heads=8,
        dropout=0.1,
        max_len=128,
        freeze_encoder=False,
    ):
        super().__init__()
        self.bert = BertModel.from_pretrained(bert_model_name)
        if freeze_encoder:
            self.bert.requires_grad_(False)

        hidden_size = self.bert.config.hidden_size
        vocab_size = self.bert.config.vocab_size

        self.embedding = nn.Embedding(vocab_size, hidden_size)
        # YOUR CODE HERE: позиционные эмбеддинги, self.decoder, self.fc_out

    @staticmethod
    def generate_square_subsequent_mask(T):
        # YOUR CODE HERE (－.－)...zzzZZZzzzZZZ
        pass

    def encode(self, input_ids, attention_mask):
        return self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state

    def decode(self, memory, attention_mask, decoder_input_ids):
        # YOUR CODE HERE: эмбеддинги + позиции, маски, декодер, fc_out
        logits = ...
        return logits

    def forward(self, input_ids, attention_mask, decoder_input_ids):
        memory = self.encode(input_ids, attention_mask)
        return self.decode(memory, attention_mask, decoder_input_ids)

    @torch.no_grad()
    def generate(self, input_ids, attention_mask, tokenizer, max_len=50):
        self.eval()
        memory = self.encode(input_ids, attention_mask)
        decoder_input_ids = torch.full(
            (input_ids.size(0), 1), BOS_ID, dtype=torch.long, device=input_ids.device
        )

        for _ in range(max_len):
            logits = self.decode(memory, attention_mask, decoder_input_ids)
            # Выберите следующий токен, допишите его в decoder_input_ids,
            # остановитесь, если сгенерирован EOS_ID
            # YOUR CODE HERE (－.－)...zzzZZZzzzZZZ
            pass

        return tokenizer.batch_decode(decoder_input_ids, skip_special_tokens=True)

In [ ]:
model = BertSummarizer().to(device)
model

Посмотрим на генерацию без обучения:

In [ ]:
eval_sample = next(iter(eval_dataloader))
model.generate(
    eval_sample["input_ids"][:1].to(device),
    eval_sample["attention_mask"][:1].to(device),
    tokenizer,
)

## Часть 3. Обучение модели (1 балл)

**Задание 3 (1 балл).** Реализуйте цикл обучения модели.

- 0.25 балла за простейший рабочий цикл;
- 0.5 балла за графики функции потерь и метрик на обучении и валидации;
- 0.25 балла за логирование в TensorBoard или WandB.

Ниже приведен один шаг обучения. Обратите внимание на teacher forcing: декодер получает на вход `labels[:, :-1]` и учится предсказывать `labels[:, 1:]`, то есть следующий токен. Паддинг в функции потерь не учитывается благодаря `ignore_index`.

In [ ]:
def train_step(model, batch, optimizer, criterion):
    model.train()
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    decoder_input_ids = labels[:, :-1]
    target = labels[:, 1:]

    optimizer.zero_grad()
    logits = model(input_ids, attention_mask, decoder_input_ids)
    loss = criterion(logits.reshape(-1, logits.size(-1)), target.reshape(-1))
    loss.backward()
    optimizer.step()

    return loss.item()


criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

In [ ]:
# YOUR CODE HERE (－.－)...zzzZZZzzzZZZ

## Часть 4. Метрики качества (1 балл)

**Задание 4 (1 балл).** Реализуйте функцию для подсчета метрик качества суммаризации, по 0.33 балла за каждую метрику:
 1. [ROUGE](https://huggingface.co/spaces/evaluate-metric/rouge);
 2. [BLEU](https://huggingface.co/spaces/evaluate-metric/bleu);
 3. [BERTScore](https://huggingface.co/spaces/evaluate-metric/bertscore).

Метрики считайте по сгенерированным моделью текстам (через `generate`), а не по выходам с teacher forcing.

Осторожно: стандартный токенизатор ROUGE выкидывает все символы, кроме латиницы и цифр, и для русского текста вернет нули. Передайте в `compute` свой `tokenizer` (например, разбиение по пробелам) и проверьте на паре строк, что метрика считается. Для BERTScore укажите `lang="ru"`.

In [ ]:
def compute_metrics(predictions, references):
    # YOUR CODE HERE (－.－)...zzzZZZzzzZZZ
    pass


def evaluation(model, dataloader, n_batches=None):
    # YOUR CODE HERE (－.－)...zzzZZZzzzZZZ
    pass

## Часть 5. Обучение и сравнение (0.5 балла)

**Задание 5 (0.5 балла).** Обучите модель, сохраните лучшую по валидации версию (`torch.save(model.state_dict(), ...)`) и покажите несколько примеров генерации рядом с эталонными краткими содержаниями.

Для сравнения посчитайте те же метрики для какой-нибудь ruT5 из 🤗 без дообучения на gazeta. Мы намеренно даем бейзлайн именно в таком виде. Хорошо ли ваша модель смотрится на его фоне?

In [ ]:
# YOUR CODE HERE (－.－)...zzzZZZzzzZZZ

In [ ]:
baseline = AutoModelForSeq2SeqLM.from_pretrained("YOUR MODEL")
summary = ...  # YOUR CODE HERE

**Ответ:** ...

## Часть 6. Менее жадные стратегии генерации (4 балла)

Всегда ли выбор самого вероятного токена на каждом шаге это лучшая стратегия генерации?

<details>
    <summary>Спойлер</summary>
    <p>Нет</p>
</details>

**Сравнение стратегий генерации:**

| Стратегия | Описание | Плюсы и минусы |
| --- | --- | --- |
| Greedy search | На каждом шаге берем самый вероятный токен. | **+** Просто и быстро. <br> **-** Часто дает повторы и бессвязный текст. |
| Сэмплирование с температурой | Сэмплируем токен из softmax(logits / T). Чем выше T, тем случайнее. | **+** Разнообразие. <br> **-** При высокой температуре текст разваливается. |
| Top-k sampling | Сэмплируем только из k самых вероятных токенов. | **+** Разнообразие без совсем маловероятных токенов. <br> **-** Фиксированное k плохо подходит и для уверенных, и для неуверенных шагов. |
| Nucleus (top-p) sampling | Сэмплируем из минимального набора самых вероятных токенов с суммарной вероятностью не меньше p. | **+** Размер набора адаптируется к распределению. <br> **-** p нужно подбирать. |
| Beam search | Храним k самых вероятных префиксов (k это ширина луча) и на каждом шаге продолжаем их. | **+** Находит более вероятные последовательности, чем greedy. <br> **-** Мало разнообразия, склонен к шаблонным ответам. |
| Нормировка на длину | Делим лог-вероятность гипотезы на длину в некоторой степени. | **+** Убирает перекос в сторону коротких гипотез. <br> **-** Степень нужно подбирать. |
| Stochastic beam search | Вносит случайность в выбор k гипотез в beam search. | **+** Больше разнообразия. <br> **-** Сложно балансировать разнообразие и качество. |
| Minimum Bayes Risk (MBR) | Из множества гипотез выбираем ту, что минимизирует ожидаемые потери относительно остальных. | **+** Оптимизирует выбранную метрику. <br> **-** Дорого и требует хорошей функции потерь. |

Полезные ссылки:
- [документация по стратегиям генерации в 🤗](https://huggingface.co/docs/transformers/generation_strategies);
- [The Curious Case of Neural Text Degeneration](https://arxiv.org/abs/1904.09751), статья про top-p.

Реализуйте стратегии сами, не через `generate` из `transformers`. Добавьте в `generate` параметр, выбирающий стратегию, или напишите отдельные функции.

**Задание 6.1 (1 балл).** Реализуйте top-k сэмплирование с температурой.

In [ ]:
# YOUR CODE HERE (－.－)...zzzZZZzzzZZZ

**Задание 6.2 (1 балл).** Реализуйте nucleus (top-p) сэмплирование.

In [ ]:
# YOUR CODE HERE (－.－)...zzzZZZzzzZZZ

**Задание 6.3 (2 балла).** Реализуйте beam search с нормировкой на длину. Достаточно поддержать батч размера 1.

In [ ]:
# YOUR CODE HERE (－.－)...zzzZZZzzzZZZ

Сравните greedy, top-k, top-p и beam search на одной и той же подвыборке валидации по метрикам из части 4 и глазами на нескольких примерах. Для сэмплирующих стратегий зафиксируйте seed. Получилось ли улучшить генерацию?

In [ ]:
# YOUR CODE HERE (－.－)...zzzZZZzzzZZZ

**Ответ:** ...

## Бонус (1 балл)

- Возьмите только декодерную часть модели (без BERT) и сделайте из нее decoder-only модель: на вход подается текст новости, разделитель и краткое содержание, функция потерь считается только по краткому содержанию.
- Поправьте цикл обучения и обучите модель в сопоставимом бюджете.
- Сравните качество генерации по метрикам и глазами.
- Ответьте на вопрос «Дает ли encoder-decoder архитектура заметный прирост качества в этой задаче?» и подтвердите ответ экспериментами.

In [ ]:
# YOUR CODE HERE (－.－)...zzzZZZzzzZZZ

**Ответ:** ...

### Источники и генеративные модели

Перечислите источники, которыми пользовались, а также модели и промпты, если использовали генеративный ИИ. Опишите, насколько это помогло.

**Ответ:** ...

Вставьте что угодно, описывающее ваши впечатления от этого задания: